In [64]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Configuración de rutas (asegúrate de que ROOT suba a la raíz)
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from config.paths import RAW_DIR, PROCESSED_DIR

YEARS = [2018, 2024]

# comentario 

In [75]:
in_kind_transfers = pd.read_csv(PROCESSED_DIR / "in_kind_transfers.csv",
                        dtype={'folioviv': str, 'foliohog': str, 'numren': int})


iva_hogar = pd.read_csv(PROCESSED_DIR / "iva_hogar.csv",
                        dtype={'folioviv': str, 'foliohog': str})


income_fiscal_incidence_base = pd.read_csv(PROCESSED_DIR / "income_fiscal_incidence_base.csv",
                        dtype={'folioviv': str, 'foliohog': str, 'numren': int})


demographic_household = pd.read_csv(PROCESSED_DIR / "demographic_household.csv",
                        dtype={'folioviv': str, 'foliohog': str})




/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_75403/2214915603.py:1: DtypeWarning: Columns (0: etnia) have mixed types. Specify dtype option on import or set low_memory=False.
  in_kind_transfers = pd.read_csv(PROCESSED_DIR / "in_kind_transfers.csv",


In [77]:
import pandas as pd

# ==========================================================
# 1. Número de integrantes por hogar (según income_fiscal_incidence_base)
# ==========================================================

print("\n--- Paso 1: household_size ---")
print(f"Tamaño de income_fiscal_incidence_base: {len(income_fiscal_incidence_base)}")

household_size = (
    income_fiscal_incidence_base
    .groupby(["folioviv", "foliohog", "year"])
    .size()
    .reset_index(name="household_size")
)

print(f"Hogares encontrados: {len(household_size)}")

household_size


--- Paso 1: household_size ---
Tamaño de income_fiscal_incidence_base: 384799
Hogares encontrados: 165949


,folioviv,foliohog,year,household_size
0,100001901,1,2024,2
1,100001902,1,2024,2
2,100001904,1,2024,2
3,100001905,1,2024,3
4,1000021401,1,2018,3
...,...,...,...,...
165944,963643714,1,2024,5
165945,963643715,1,2024,3
165946,963643716,1,2024,3
165947,963643717,1,2024,3


In [78]:
check = iva_hogar.merge(
    household_size,
    on=["folioviv", "foliohog", "year"],
    how="left",
    indicator=True
)

print(check["_merge"].value_counts())

_merge
both          113421
left_only      52529
right_only         0
Name: count, dtype: int64


In [79]:
check["estado"] = check["folioviv"].str[:2]

print(check.groupby(["estado", "_merge"]).size().unstack(fill_value=0))

_merge  left_only  both
estado                 
01           5212     0
02           7362     0
03           5181     0
04           4053     0
05           7295     0
06           6133     0
07           3982     0
08           8536     0
09           4745     0
10              3  5053
11              0  5774
12              1  4471
13              0  4193
14              3  4750
15              1  6424
16              0  4002
17              3  4570
18              1  3867
19              8  6829
20              1  4919
21              0  4031
22              0  6814
23              1  4173
24              0  4825
25              0  6281
26              5  4731
27              0  4084
28              0  4232
29              1  4101
30              0  5319
31              2  5264
32              0  4714


In [80]:
print(iva_hogar["folioviv"].head(20))
print(household_size["folioviv"].head(20))

0     0100013601
1     0100013602
2     0100013603
3     0100013604
4     0100013606
5     0100026701
6     0100026703
7     0100026704
8     0100026706
9     0100027201
10    0100027202
11    0100027204
12    0100027205
13    0100045501
14    0100045502
15    0100045503
16    0100045505
17    0100045506
18    0100054301
19    0100054302
Name: folioviv, dtype: str
0      100001901
1      100001902
2      100001904
3      100001905
4     1000021401
5     1000021402
6     1000021404
7     1000021405
8      100002501
9      100002502
10     100002504
11     100002505
12     100002506
13    1000035901
14    1000035902
15    1000035903
16    1000035904
17    1000035905
18    1000035906
19    1000036001
Name: folioviv, dtype: str


In [81]:
print(iva_hogar["folioviv"].str.len().value_counts())
print(household_size["folioviv"].str.len().value_counts())

folioviv
10    165950
Name: count, dtype: int64
folioviv
10    113454
9      52495
Name: count, dtype: int64


In [66]:
import pandas as pd

# ==========================================================
# 1. Número de integrantes por hogar (según income_fiscal_incidence_base)
# ==========================================================

print("\n--- Paso 1: household_size ---")
print(f"Tamaño de income_fiscal_incidence_base: {len(income_fiscal_incidence_base)}")

household_size = (
    income_fiscal_incidence_base
    .groupby(["folioviv", "foliohog", "year"])
    .size()
    .reset_index(name="household_size")
)

print(f"Hogares encontrados: {len(household_size)}")

# ==========================================================
# 2. Convertir IVA hogar a IVA per cápita
# ==========================================================

print("\n--- Paso 2: IVA per cápita ---")
print(f"Tamaño de iva_hogar antes: {len(iva_hogar)}")

iva_hogar = iva_hogar.merge(
    household_size,
    on=["folioviv", "foliohog", "year"],
    how="inner"
)

print(f"Tamaño de iva_hogar después del merge: {len(iva_hogar)}")

# Todas las columnas de IVA (excepto llaves) se dividen entre el tamaño del hogar
iva_cols = [
    c for c in iva_hogar.columns
    if c not in ["folioviv", "foliohog", "year", "household_size"]
]

iva_hogar[iva_cols] = iva_hogar[iva_cols].div(
    iva_hogar["household_size"],
    axis=0
)


print("Variables de IVA convertidas a nivel individuo.")

# ==========================================================
# 3. Base a nivel hogar
# ==========================================================

print("\n--- Paso 3: Base a nivel hogar ---")
print(f"Tamaño de demographic_household (antes del merge): {len(demographic_household)}")
print(f"Tamaño de iva_hogar (antes del merge): {len(iva_hogar)}")

household_base = (
    demographic_household
    .merge(
        iva_hogar,
        on=["folioviv", "foliohog", "year"],
        how="inner"
    )
)

print(f"Tamaño de household_base (después del merge): {len(household_base)}")

# ==========================================================
# 4. Llevar variables del hogar a la base individual
# ==========================================================

print("\n--- Paso 4: Base individual con variables del hogar ---")
print(f"Tamaño de income_fiscal_incidence_base (antes del merge): {len(income_fiscal_incidence_base)}")
print(f"Tamaño de household_base (antes del merge): {len(household_base)}")

individual_base = (
    income_fiscal_incidence_base
    .merge(
        household_base,
        on=["folioviv", "foliohog", "year"],
        how="inner"
    )
)

print(f"Tamaño de individual_base (después del merge): {len(individual_base)}")

# ==========================================================
# 5. Agregar transferencias en especie
# ==========================================================

print("\n--- Paso 5: Agregar transferencias en especie ---")

# Asegurar que las llaves tengan el mismo tipo
individual_base["folioviv"] = individual_base["folioviv"].astype(str)
individual_base["foliohog"] = individual_base["foliohog"].astype(str)
individual_base["numren"] = individual_base["numren"].astype(str)

in_kind_transfers["folioviv"] = in_kind_transfers["folioviv"].astype(str)
in_kind_transfers["foliohog"] = in_kind_transfers["foliohog"].astype(str)
in_kind_transfers["numren"] = in_kind_transfers["numren"].astype(str)

print(f"Tamaño de individual_base (antes del merge con transfers): {len(individual_base)}")
print(f"Tamaño de in_kind_transfers (antes del merge): {len(in_kind_transfers)}")

# Merge
individual_base = individual_base.merge(
    in_kind_transfers,
    on=["folioviv", "foliohog", "numren", "year"],
    how="inner"
)

print(f"Tamaño de individual_base (después del merge con transfers): {len(individual_base)}")

# ==========================================================
# 6. Conteo por año (2018 y 2024)
# ==========================================================

print("\n--- Conteo de registros por año en la base final ---")
print(individual_base['year'].value_counts().sort_index())

# Si quieres ver específicamente 2018 y 2024:
print(f"\nRegistros para 2018: {len(individual_base[individual_base['year'] == 2018])}")
print(f"Registros para 2024: {len(individual_base[individual_base['year'] == 2024])}")

# ==========================================================
# Resultado
# ==========================================================

print("\n--- Muestra de la base final ---")
print(individual_base.head())

individual_base.to_csv(PROCESSED_DIR / "base_final.csv", index=False)
print("Saved base_final.csv")


--- Paso 1: household_size ---
Tamaño de income_fiscal_incidence_base: 384799
Hogares encontrados: 165949

--- Paso 2: IVA per cápita ---
Tamaño de iva_hogar antes: 165950
Tamaño de iva_hogar después del merge: 113421
Variables de IVA convertidas a nivel individuo.

--- Paso 3: Base a nivel hogar ---
Tamaño de demographic_household (antes del merge): 166061
Tamaño de iva_hogar (antes del merge): 113421
Tamaño de household_base (después del merge): 113421

--- Paso 4: Base individual con variables del hogar ---
Tamaño de income_fiscal_incidence_base (antes del merge): 384799
Tamaño de household_base (antes del merge): 113421
Tamaño de individual_base (después del merge): 267450

--- Paso 5: Agregar transferencias en especie ---
Tamaño de individual_base (antes del merge con transfers): 267450
Tamaño de in_kind_transfers (antes del merge): 577804
Tamaño de individual_base (después del merge con transfers): 267450

--- Conteo de registros por año en la base final ---
year
2018    127600


In [67]:
demographic_household

,folioviv,foliohog,ubica_geo,state,municipality,upm,est_dis,factor,year
0,0100013601,1,1001,1,1,1,2,179,2018
1,0100013602,1,1001,1,1,1,2,179,2018
2,0100013603,1,1001,1,1,1,2,179,2018
3,0100013604,1,1001,1,1,1,2,179,2018
4,0100013606,1,1001,1,1,1,2,179,2018
...,...,...,...,...,...,...,...,...,...
166056,3260593814,1,32052,32,52,10593,680,183,2024
166057,3260593815,1,32052,32,52,10593,680,183,2024
166058,3260593816,1,32052,32,52,10593,680,183,2024
166059,3260593817,1,32052,32,52,10593,680,183,2024


In [68]:
household_base

,folioviv,foliohog,ubica_geo,state,municipality,upm,est_dis,factor,year,gasto_con_iva,gasto_sin_iva,iva_pagado,gasto_total,gasto_gas,gasto_electricity,household_size
0,1000021401,1,10005,10,5,2877,146,326,2018,32232.486667,1694.2300,4445.860230,36865.613333,6580.643333,225.0,3
1,1000021402,1,10005,10,5,2877,146,326,2018,24125.790000,7417.2550,3327.695172,32325.645000,3629.030000,202.5,2
2,1000021404,1,10005,10,5,2877,146,326,2018,7196.790000,11127.6400,992.660690,23550.230000,0.000000,333.0,1
3,1000021405,1,10005,10,5,2877,146,326,2018,14548.627500,3013.5125,2006.707241,17562.140000,0.000000,82.5,4
4,1000035901,1,10005,10,5,2878,145,265,2018,16505.456667,3029.9700,2276.614713,27043.833333,2322.580000,301.0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113416,3260593814,1,32052,32,52,10593,680,183,2024,8271.985000,1478.5450,1140.963448,9750.530000,1500.000000,90.0,2
113417,3260593815,1,32052,32,52,10593,680,183,2024,17894.580000,7262.7950,2468.217931,27407.370000,7500.000000,225.0,2
113418,3260593816,1,32052,32,52,10593,680,183,2024,4123.930000,1519.9100,568.817931,5741.663333,2000.000000,140.0,3
113419,3260593817,1,32052,32,52,10593,680,183,2024,8599.910000,3869.9600,1186.194483,12929.650000,3600.000000,180.0,1


In [69]:
iva_hogar

,folioviv,foliohog,gasto_con_iva,gasto_sin_iva,iva_pagado,gasto_total,gasto_gas,gasto_electricity,year,household_size
0,1000021401,1,32232.486667,1694.2300,4445.860230,36865.613333,6580.643333,225.0,2018,3
1,1000021402,1,24125.790000,7417.2550,3327.695172,32325.645000,3629.030000,202.5,2018,2
2,1000021404,1,7196.790000,11127.6400,992.660690,23550.230000,0.000000,333.0,2018,1
3,1000021405,1,14548.627500,3013.5125,2006.707241,17562.140000,0.000000,82.5,2018,4
4,1000035901,1,16505.456667,3029.9700,2276.614713,27043.833333,2322.580000,301.0,2018,3
...,...,...,...,...,...,...,...,...,...,...
113416,3260593814,1,8271.985000,1478.5450,1140.963448,9750.530000,1500.000000,90.0,2024,2
113417,3260593815,1,17894.580000,7262.7950,2468.217931,27407.370000,7500.000000,225.0,2024,2
113418,3260593816,1,4123.930000,1519.9100,568.817931,5741.663333,2000.000000,140.0,2024,3
113419,3260593817,1,8599.910000,3869.9600,1186.194483,12929.650000,3600.000000,180.0,2024,1


In [70]:
base_final = pd.read_csv(PROCESSED_DIR / "base_final.csv",
                        dtype={'folioviv': str, 'foliohog': str, 'numren': str})

base_final["state"].value_counts()

base_final

/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_75403/3165557407.py:1: DtypeWarning: Columns (0: etnia) have mixed types. Specify dtype option on import or set low_memory=False.
  base_final = pd.read_csv(PROCESSED_DIR / "base_final.csv",


,folioviv,foliohog,numren,year,net_trabajo1_ann,net_trabajo2_ann,isr_total,formal_trabajo_1,formal_trabajo_2,mi_trabajo_1,...,iva_pagado,gasto_total,gasto_gas,gasto_electricity,household_size,sexo,edad,etnia,education,health
0,1000021401,1,2,2018,322826.04,0.00,62521.198849,1.0,0.0,385347.238849,...,4445.860230,36865.613333,6580.643333,225.0,3,1,59,2,0,1
1,1000021401,1,3,2018,0.00,89021.68,6598.270485,0.0,1.0,0.000000,...,4445.860230,36865.613333,6580.643333,225.0,3,1,25,2,0,1
2,1000021401,1,1,2018,0.00,0.00,0.000000,0.0,0.0,0.000000,...,4445.860230,36865.613333,6580.643333,225.0,3,2,55,2,0,1
3,1000021402,1,1,2018,0.00,0.00,0.000000,0.0,0.0,0.000000,...,3327.695172,32325.645000,3629.030000,202.5,2,2,41,1,0,1
4,1000021402,1,4,2018,0.00,0.00,0.000000,0.0,0.0,0.000000,...,3327.695172,32325.645000,3629.030000,202.5,2,1,35,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267445,3260593816,1,3,2024,32262.28,0.00,0.000000,0.0,0.0,24158.561919,...,568.817931,5741.663333,2000.000000,140.0,3,1,21,2.0,0,0
267446,3260593816,1,4,2024,0.00,0.00,0.000000,0.0,0.0,0.000000,...,568.817931,5741.663333,2000.000000,140.0,3,1,15,2.0,1,0
267447,3260593817,1,2,2024,0.00,0.00,0.000000,0.0,0.0,0.000000,...,1186.194483,12929.650000,3600.000000,180.0,1,2,54,2.0,0,1
267448,3260593818,1,1,2024,28524.56,0.00,0.000000,0.0,0.0,21359.691533,...,332.148276,8406.885000,0.000000,337.5,2,2,63,2.0,0,0
